In [62]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import holidays

In [63]:
# loading and cleaning interval data:
def clean_interval(portfolio):
    df = pd.read_csv(f'./data/{portfolio} - Interval.csv', index_col=0)

    df['Interval'] = df['Interval'].apply(lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x))
    interval = pd.date_range('2025-04-01 00:00', '2025-06-30 23:30', freq='30min')
    time_inters = pd.DataFrame({
        'Month' : interval.month_name(),
        'Day' : interval.day,
        'Interval' : interval.strftime('%H:%M:%S')
    }) # Creates proper intervals to make sure that the intervals are correct
    df = time_inters.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    #time features
    df['Date'] = pd.to_datetime(df['Month'] + " " + df['Day'].astype(str) + " 2025")
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['IntervalInd'] = df['Interval'].apply(lambda x: 2*int(x.split(':')[0]) + int(x.split(':')[1])//30)
    df['month_num'] = df['Date'].dt.month
    df['week'] = (df['Day']-1)//7 + 1
    df['weekend'] = (df['DayOfWeek'] >=5).astype(int)

    targets = ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']
    
    for target in targets:
        df[target] = df.groupby(['Month', 'Day'])[target].transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        df[target] = df.groupby(['DayOfWeek', "IntervalInd"])[target].transform(lambda x: x.fillna(x.median()))
        df[target] = df[target].fillna(df[target].median()).clip(lower=0)
    
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1) # Should be between 0 and 1

    # convert hour and day of week to cyclical features
    df['hour_sin'] = np.sin(2*np.pi*df['IntervalInd']/48)
    df['hour_cos'] = np.cos(2*np.pi*df['IntervalInd']/48)
    df['dow_sin'] = np.sin(2*np.pi*df['IntervalInd']/48)
    df['dow_cos'] = np.cos(2*np.pi*df['IntervalInd']/48)

    return df

In [64]:
clean_interval('A').columns

Index(['Month', 'Day', 'Interval', 'Service Level', 'Call Volume',
       'Abandoned Calls', 'Abandoned Rate', 'CCT', 'Date', 'DayOfWeek',
       'IntervalInd', 'month_num', 'week', 'weekend', 'hour_sin', 'hour_cos',
       'dow_sin', 'dow_cos'],
      dtype='object')

In [65]:
def clean_daily(portfolio):
    df = pd.read_csv(f'./data/{portfolio} - Daily.csv', index_col=0)

    #time features
    df['Date'] = pd.to_datetime(df['Date'].str[0:8], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['Day'] = df['Date'].dt.day
    df['week'] = (df['Day'] - 1)//7 + 1
    df['month_num'] = df['Date'].dt.month
    df['weekend'] = (df['DayOfWeek'] >=5 ).astype(int)

    # clean missing data
    targets = ['Call Volume', 'CCT', 'Abandon Rate']

    for target in targets:
        df[target] = df.groupby('DayOfWeek')[target].transform(lambda x: x.fillna(x.median()))
        df[target] = df.groupby('month_num')[target].transform(lambda x: x.fillna(x.median()))
        df[target] = df[target].fillna(df[target].median())

    df[['Call Volume', 'CCT', 'Abandon Rate']] = df[['Call Volume', 'CCT', 'Abandon Rate']].clip(lower=0)

    # cyclical features for day of week and month
    df['dow_sin']   = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)  

    # lag features - shift 364 days
    df['cv_lag_364']  = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)

    # add august mean
    august = df[df['month_num'] == 8]
    df['aug_cv_mean']  = august['Call Volume'].mean()
    df['aug_cct_mean'] = august['CCT'].mean()
    df['aug_abd_mean'] = august['Abandon Rate'].mean()

    # august mean by day of the week
    august_dow = august.groupby('DayOfWeek')[targets].mean()
    august_dow.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    df = df.merge(august_dow, on='DayOfWeek', how='left')

    # Fill in nans
    for metric in ['aug_cv_mean','aug_cct_mean','aug_abd_mean', 'aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']:
        df[metric] = df[metric].fillna(df[metric].median())

    return df

In [66]:
clean_daily('A').columns

Index(['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate',
       'DayOfWeek', 'Day', 'week', 'month_num', 'weekend', 'dow_sin',
       'dow_cos', 'month_sin', 'month_cos', 'cv_lag_364', 'cct_lag_364',
       'abd_lag_364', 'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
       'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'],
      dtype='object')

In [67]:
# Pipeline

portfolios = ['A', 'B', 'C', 'D']
features = ['DayOfWeek', 'month_num', 'Day', 'week', 'weekend', 
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364',
    'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
    'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'
]
targets = ['Call Volume', 'CCT', 'Abandon Rate']

august_days = pd.date_range('2026-08-01', '2026-08-31', freq='D')
forecasts = []

for p in portfolios:
    # Build models
    df_d = clean_daily(p)
    df_iv = clean_interval(p)
    df_iv = df_iv.merge(df_d[['Date', 'Call Volume']].rename(columns={'Call Volume' : 'daily_cv'}), on='Date', how='left')

    d_train = df_d.dropna(subset=features + ['Call Volume'])
    models = {}
    for target in targets:
        model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=5, random_state=42)
        model.fit(d_train[features], d_train[target])
        models[target] = model

    # August 2026 intervals
    august = pd.DataFrame({
        'Date' : august_days,
        'DayOfWeek' : august_days.dayofweek,
        'month_num' : august_days.month,
        'Day' :  august_days.day,
        'week' : (august_days.day - 1) // 7 + 1,
        'weekend' : (august_days.dayofweek >= 5).astype(int),
    })

    #cyclical features
    august['dow_sin']   = np.sin(2 * np.pi * august['DayOfWeek'] / 7)
    august['dow_cos']   = np.cos(2 * np.pi * august['DayOfWeek'] / 7)
    august['month_sin'] = np.sin(2 * np.pi * august['month_num'] / 12)
    august['month_cos'] = np.cos(2 * np.pi * august['month_num'] / 12)

    # lags
    aug_2025 = df_d[
        (df_d['month_num'] == 8) & (df_d['Date'].dt.year == 2025)
    ].reset_index(drop=True)
    august['cv_lag_364']  = aug_2025['Call Volume'].values
    august['cct_lag_364'] = aug_2025['CCT'].values
    august['abd_lag_364'] = aug_2025['Abandon Rate'].values

    #baseline
    aug = df_d[df_d['month_num'] == 8]
    august['aug_cv_mean']  = aug['Call Volume'].mean()
    august['aug_cct_mean'] = aug['CCT'].mean()
    august['aug_abd_mean'] = aug['Abandon Rate'].mean()

    aug_dow_means = aug.groupby('Day')[['Call Volume','CCT','Abandon Rate']].mean()
    aug_dow_means.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    august = august.merge(aug_dow_means, on='Day', how='left')

    for target, pred in [('Call Volume', 'pred_cv'), ('CCT', 'pred_cct'), ('Abandon Rate', 'pred_abd')]:
        august[pred] = np.maximum(0, models[target].predict(august[features]))

    august['pred_cv'] = august['pred_cv'] * 1.15 # prevent understaffing

    # Intraday Profiling
    iv_weighted = pd.concat([df_iv, df_iv[df_iv['month_num'] == 6]], ignore_index=True)
    iv_weighted['cv_share'] = iv_weighted['Call Volume']/iv_weighted['daily_cv'].clip(lower=1)
    profile_cv = iv_weighted.groupby(['DayOfWeek', 'IntervalInd'])['cv_share'].mean()

    dow_sums = profile_cv.groupby(level='DayOfWeek').transform('sum')
    profile_cv = profile_cv / dow_sums.clip(lower=1e-9)

    profile_cct = iv_weighted.groupby(['DayOfWeek', 'IntervalInd'])['CCT'].mean()

    profile_abd = iv_weighted.groupby(['DayOfWeek', 'IntervalInd'])['Abandoned Rate'].mean()

    # generate forecasts
    rows = []
    for i, row in august.iterrows():
        dow = int(row['DayOfWeek'])
        day = int(row['Day'])

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_share = profile_cv.get((dow, iod), 1/48)
            cv_pred  = max(0, cv_share * row['pred_cv'])
            cct_pred = max(0, profile_cct.get((dow, iod), row['pred_cct']))
            abd_pred = float(np.clip(profile_abd.get((dow, iod), 0.01), 0, 1))

            rows.append({
                'Month':                 'August',
                'Day':                   day,
                'Interval':              f"{h}:{mn:02d}",
                f'Calls_Offered_{p}':    round(cv_pred, 2),
                f'Abandoned_Calls_{p}':  round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':   round(abd_pred, 6),
                f'CCT_{p}':              round(cct_pred, 2),
            })

    forecasts.append(pd.DataFrame(rows))
    print(f"{p}: {len(rows)} rows forecasted!")


A: 1488 rows forecasted!
B: 1488 rows forecasted!
C: 1488 rows forecasted!
D: 1488 rows forecasted!


In [68]:
# Combining and shipping

# start with intervals, then merge A, B, C, D
result = forecasts[0][['Month', 'Day', 'Interval']]
cols = ['Month', 'Day', 'Interval']
for i in range(len(portfolios)):
    p = portfolios[i]
    print(p)
    result = result.merge(forecasts[i][['Month', 'Day', 'Interval', f'Calls_Offered_{p}', f'Abandoned_Calls_{p}', f'Abandoned_Rate_{p}', f'CCT_{p}']],
    on=['Month', 'Day', 'Interval'], how='left')
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

    cols += [f'Calls_Offered_{p}', f'Abandoned_Calls_{p}', f'Abandoned_Rate_{p}', f'CCT_{p}']

result = result[cols]

result.to_csv('./August 2026 Forecasts/final_forecast.csv', index=False)

print(f'Saved to ./August 2026 Forecasts/final_forecast.csv')
print(f"Shape: {result.shape}")
result.head()

A
B
C
D
Saved to ./August 2026 Forecasts/final_forecast.csv
Shape: (1488, 19)


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,0:00,8.03,0.39,0.048529,351.13,53.80,1.35,0.025041,294.88,122.15,1.71,0.013976,322.53,54.76,1.37,0.024953,300.69
1,August,1,0:30,6.36,0.00,0.000000,358.94,32.42,0.00,0.000000,334.86,91.01,0.22,0.002400,327.42,37.02,0.37,0.010112,308.27
2,August,1,1:00,5.03,0.26,0.050976,352.08,14.07,0.89,0.063400,365.35,68.40,1.10,0.016124,304.40,30.26,0.08,0.002559,311.04
3,August,1,1:30,3.06,0.21,0.068624,295.89,7.04,0.18,0.026141,282.17,59.74,0.73,0.012182,333.25,21.46,0.31,0.014324,307.24
4,August,1,2:00,2.63,0.15,0.058824,412.41,6.87,0.20,0.029412,310.25,34.66,0.62,0.017918,308.02,13.93,0.15,0.010812,315.72
